# Exercício 11 — Regressão com scikit-learn

Este exercício repete o pipeline da aula na **sua coleta de rede social** (a mesma que você usou nas Aulas 5 a 7). Você vai treinar um modelo que estima a taxa de engajamento de um post a partir de características que existem *antes* da publicação, e comparar esse modelo com um chute burro.

Não é para inventar coleta nova. Copie a sua `exportacao.csv` das aulas anteriores para `dados/exportacao.csv` dentro desta pasta.

Antes de tudo, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`, com o SEU nome), numa pasta `11-regressao-scikit-learn` dentro de `projetos/`.

## Preparação do ambiente

Este notebook mora numa pasta própria (dentro da sua pasta de entregas). Crie o ambiente e instale as dependências dentro dela.

No Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, use `pip install -r requirements.txt` com o ambiente já ativado.

## Parte 0 — Dados

**Fonte dos dados:** Exportação do TikTok via Zeeschuimer, Aula 7.

**Pergunta que este modelo tenta responder (preencha):**

> Dá para estimar a taxa de engajamento de um post da minha coleta a partir de características do autor, do texto e do horário?

## Parte 1 — Carregar e construir o alvo

Esse trecho já vem pronto (é o mesmo da aula). Ajuste os nomes de coluna se a sua plataforma usar nomes diferentes de `likes`, `comments`, `shares`, `plays`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("dados/exportacao.csv", sep=";")
df = df.drop_duplicates()
df = df[df["plays"] > 0].copy()

df["taxa_engajamento"] = (
    df["likes"] + df["comments"] + df["shares"]
) / df["plays"]

y = df["taxa_engajamento"]
print(f"Posts na base: {len(df)}")
y.describe()

Posts na base: 1138


count    1138.000000
mean        0.038749
std         0.047725
min         0.000000
25%         0.009511
50%         0.019851
75%         0.048251
max         0.451807
Name: taxa_engajamento, dtype: float64

## Parte 2 — Construir as suas features

**Regra de ouro: nada de vazamento.** `likes`, `comments`, `shares` e `plays` NÃO entram no `X` — eles são o alvo.

Monte um `X` com **pelo menos três features derivadas por você**. Pode reaproveitar as da aula (nº de hashtags, tamanho da legenda, nº de emojis, hora, dia da semana, seguidores do autor) e/ou inventar outras que façam sentido para a sua coleta (tem menção `@`? tem link? tem ponto de interrogação? é fim de semana?).

In [2]:
import re  # para achar emojis no texto

# alvo separado
y = df["taxa_engajamento"]

# --- features derivadas, todas "de antes da publicação" ---
X = pd.DataFrame(index=df.index)  # tabela de features vazia, mesmo índice de df

# do autor
X["seguidores_autor"] = df["author_followers"]  # tamanho da audiência de quem postou
X["videos_autor"] = df["author_videos"]  # quantos vídeos o autor já publicou no total

# do texto (coluna body = legenda do post)
X["tam_legenda"] = df["body"].fillna("").str.len()  # nº de caracteres da legenda
padrao_emoji = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]")  # faixa Unicode de emojis
X["n_emojis"] = df["body"].fillna("").apply(lambda s: len(padrao_emoji.findall(s)))

# das hashtags (string separada por vírgula, pode ser vazia)
X["n_hashtags"] = df["hashtags"].fillna("").apply(
    lambda s: 0 if s == "" else len(s.split(","))
)

# do horário
momento = pd.to_datetime(df["timestamp"])  # converte o texto de data para datetime
X["hora"] = momento.dt.hour  # 0 a 23
X["dia_semana"] = momento.dt.dayofweek  # 0 = segunda ... 6 = domingo

X.head() 

print("Features criadas:", list(X.columns))
X.head()

Features criadas: ['seguidores_autor', 'videos_autor', 'tam_legenda', 'n_emojis', 'n_hashtags', 'hora', 'dia_semana']


,seguidores_autor,videos_autor,tam_legenda,n_emojis,n_hashtags,hora,dia_semana
0,2648,255,28,1,0,19,3
1,1300000,3430,25,0,4,16,2
2,185300,2464,15,0,1,21,2
3,1708,32,44,0,1,11,0
4,12900,146,71,0,3,15,0


**Confira antes de seguir:** todas as colunas de `X` são numéricas? Nenhuma é `likes`/`comments`/`shares`/`plays`? Nenhuma tem valor ausente? (`X.isna().sum()` e `X.dtypes`)

In [3]:
print(X.dtypes)
print()
print("valores ausentes por coluna:")
print(X.isna().sum())

seguidores_autor    int64
videos_autor        int64
tam_legenda         int64
n_emojis            int64
n_hashtags          int64
hora                int32
dia_semana          int32
dtype: object

valores ausentes por coluna:
seguidores_autor    0
videos_autor        0
tam_legenda         0
n_emojis            0
n_hashtags          0
hora                0
dia_semana          0
dtype: int64


## Parte 3 — Treino, teste e modelo bobo

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# modelo bobo: prever sempre a média do treino
previsao_boba = np.full(len(y_teste), y_treino.mean())
mae_bobo = mean_absolute_error(y_teste, previsao_boba)
r2_bobo = r2_score(y_teste, previsao_boba)

# regressão linear
modelo_linear = LinearRegression()
modelo_linear.fit(X_treino, y_treino)
previsoes = modelo_linear.predict(X_teste)
mae_linear = mean_absolute_error(y_teste, previsoes)
r2_linear = r2_score(y_teste, previsoes)

print(f"Modelo bobo:       MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:  MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")

Modelo bobo:       MAE = 0.0324   R2 = -0.000
Regressão linear:  MAE = 0.0330   R2 = -0.007


## Parte 4 — Uma árvore, para comparar

Complete a célula: treine uma `DecisionTreeRegressor` (use `max_depth=5, random_state=42`), preveja em `X_teste` e calcule MAE e R². Imprima os três modelos lado a lado (bobo, linear, árvore).

In [ ]:
from sklearn.tree import DecisionTreeRegressor

from sklearn.tree import DecisionTreeRegressor

modelo_arvore = DecisionTreeRegressor(max_depth=5, random_state=42)
modelo_arvore.fit(X_treino, y_treino)
previsoes_arvore = modelo_arvore.predict(X_teste)

mae_arvore = mean_absolute_error(y_teste, previsoes_arvore)
r2_arvore = r2_score(y_teste, previsoes_arvore)

print(f"Modelo bobo:        MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:   MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")
print(f"Árvore (prof. 5):   MAE = {mae_arvore:.4f}   R2 = {r2_arvore:.3f}")

# MAE deve ser o menor possível (quanto mais próximo de 0, menos o modelo está errando). R2 deve ser o maior possível 
#(quanto mais próximo de 1, melhor o modelo está performando - é melhor do que só chutar a média, nesse caso da taxa de engajamento).

Modelo bobo:        MAE = 0.0324   R2 = -0.000
Regressão linear:   MAE = 0.0330   R2 = -0.007
Árvore (prof. 5):   MAE = 0.0323   R2 = -0.295


## Parte 5 — Ler os coeficientes

Complete: monte um DataFrame com `X.columns` e `modelo_linear.coef_`, ordenado pelo coeficiente. Olhe o sinal de cada um e escreva, no rascunho abaixo, **uma** leitura em uma frase (lembrando: associação nesta coleta, não causa).

In [6]:
# complete aqui: DataFrame com feature e coeficiente, ordenado
coeficientes = pd.DataFrame({
    "feature": X.columns,
    "coeficiente": modelo_linear.coef_,
}).sort_values("coeficiente")
coeficientes

,feature,coeficiente
6,dia_semana,-1.198124e-03
3,n_emojis,-2.040889e-04
2,tam_legenda,-6.254881e-05
1,videos_autor,-2.458109e-06
0,seguidores_autor,2.340655e-09
5,hora,1.182711e-03
4,n_hashtags,2.353560e-03


**Rascunho da leitura de um coeficiente (vai para o README):**

> Nesta coleta, o modelo encontrou que posts de autores com mais seguidores e que possuiam mais hashtags tiveram uma taxa de engajamento maior. Isso é uma associação nos meus dados, não quer dizer que ganhar mais seguidores ou usar mais hahstags sempre aumentará o engajamento.".

## Parte 6 — README de reprodução

Crie o arquivo `README.md` dentro de `projetos/11-regressao-scikit-learn/` na sua pasta de entregas, respondendo:

**Fonte e período dos dados:**

> Fonte: Exportação do TikTok via Zeeschuimer, Aula 7. Período: 19/08/2026

**Quantos posts entraram no modelo (depois de remover duplicata e `plays` = 0):**

> 1138 posts

**Quais features você usou, e por quê:**

> Features criadas: 'seguidores_autor', 'videos_autor', 'tam_legenda', 'n_emojis', 'n_hashtags', 'hora', 'dia_semana'. Foram utilizadas essas features, pois mesmo elas não conseguindo provar exatamente qual seria a taxa de engajamento de um post, elas conseguem chegar em uma média muito boa de qual seria aquela taxa, analisando outros posts que obteram valores parecidos nessas variáveis. Além disso, não seria ideal usar "likes", "comments", "shares" e "plays", pois eles são diretamente utilizados para fazer o cálculo do engajamento médio, ou seja, você não estaria prevendo nada pois você já teria informado os números para fazer o cálculo exato do engajamento, isto é, não seria uma previsão.

**Quais colunas você deixou de fora por vazamento, e por quê:**

> As colunas "likes", "comments", "shares" e "plays", pois elas são diretamente utilizadas para fazer o cálculo do engajamento médio, ou seja, você não estaria prevendo nada pois você já teria informado os números para fazer o cálculo exato do engajamento, isto é, não seria uma previsão.

**Resultado: MAE e R² do modelo bobo, da linear e da árvore. O seu melhor modelo bateu o bobo?**

> Modelo bobo:  MAE = 0.0324   R2 = -0.000 / Regressão linear:   MAE = 0.0330   R2 = -0.007 / Árvore (prof. 5):   MAE = 0.0323   R2 = -0.295. Neste caso, o melhor MAE alcançado foi o da Árvore, com uma diferença de 0.001 (muito pequena) para o modelo bobo. Já no R2, o melhor continuou sendo o do modelo bobo, visto que o dos algoritmos testados deu um resultado negativo mais alto.

**Uma leitura de coeficiente (associação, não causa):**

> Nesta coleta, o modelo encontrou que posts de autores com mais seguidores e que possuiam mais hashtags tiveram uma taxa de engajamento maior. Isso é uma associação nos meus dados, não quer dizer que ganhar mais seguidores ou usar mais hahstags sempre aumentará o engajamento.".

**Declaração de uso de IA:** 
> Não usei IA nesta entrega.


## Parte 7 — Conferência final

Antes de considerar a entrega concluída, confira:

- [ ] `dados/exportacao.csv` é a sua coleta das Aulas 5 a 7, e a pasta `dados/` está no `.gitignore`.
- [ ] O `X` tem pelo menos três features derivadas por você, todas numéricas, nenhuma delas `likes`/`comments`/`shares`/`plays`.
- [ ] A regressão linear e a árvore foram treinadas e comparadas com o modelo bobo.
- [ ] O README tem: fonte, nº de posts, lista de features, colunas descartadas por vazamento, os três resultados (MAE/R²) e uma leitura de coeficiente.
- [ ] Este notebook roda do início ao fim sem erro com Kernel → Restart e Run All (ou o equivalente no VS Code).
- [ ] O README registra o uso de IA (ou informa que não houve).
- [ ] Este notebook e o README estão copiados dentro de `projetos/11-regressao-scikit-learn/` na sua pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.